<a href="https://colab.research.google.com/github/abuzar01440/abuzar-portfolio/blob/main/feature_eng_and_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 602.4/602.4 kB 31.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,  classification_report, jaccard_score
from sklearn.ensemble import RandomForestClassifier
import optuna
from sklearn.model_selection import cross_val_score

In [4]:
df= pd.read_csv("/content/clean_data_after_eda.csv")
df.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000003,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,0.000011,2.896760e-06,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0


In [5]:
df.drop(["origin_up","channel_sales"],axis=1,inplace=True)

In [6]:
df.head(2)

,id,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,forecast_cons_year,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,0,...,0.000131,0.000041,0.000908,2.086294,99.530517,44.235794,2.086425,99.530558,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,0,...,0.000003,0.001218,0.000000,0.009482,0.000000,0.000000,0.009485,0.001218,0.000000,0


In [7]:
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [8]:
price_df = pd.read_csv('/content/price_data (1).csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [9]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


In [10]:
df_1=diff.join(df.set_index('id'), on='id')

In [11]:

df_1.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916,22034.0,0.0,3084.0,2010-01-19,2016-02-21,2010-01-19,2015-02-25,...,0.000011,0.000003,4.860000e-10,0.0,0.0,0.0,0.000011,0.000003,4.860000e-10,0.0
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779,4060.0,0.0,0.0,2009-08-06,2016-06-21,2013-06-21,2015-06-23,...,0.000003,0.000000,0.000000e+00,0.0,0.0,0.0,0.000003,0.000000,0.000000e+00,0.0
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000,7440.0,0.0,1062.0,2013-02-25,2016-05-05,2015-05-05,2015-02-26,...,0.000003,0.000000,0.000000e+00,0.0,0.0,0.0,0.000003,0.000000,0.000000e+00,0.0
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916,NaN,NaN,NaN,NaT,NaT,NaT,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001,11272.0,0.0,0.0,2010-03-02,2016-03-02,2010-03-02,2015-03-09,...,0.000003,0.000000,0.000000e+00,0.0,0.0,0.0,0.000003,0.000000,0.000000e+00,0.0


In [12]:
#check null values sum
df_1.isnull().sum()

,0
id,0
offpeak_diff_dec_january_energy,0
offpeak_diff_dec_january_power,0
cons_12m,1490
cons_gas_12m,1490
cons_last_month,1490
date_activ,1490
date_end,1490
date_modif_prod,1490
date_renewal,1490


In [13]:
df_1.dropna(inplace=True)

In [14]:
df_1.drop("id", inplace=True, axis=1)

In [15]:
df_1.isnull().sum()

,0
offpeak_diff_dec_january_energy,0
offpeak_diff_dec_january_power,0
cons_12m,0
cons_gas_12m,0
cons_last_month,0
date_activ,0
date_end,0
date_modif_prod,0
date_renewal,0
forecast_cons_12m,0


In [16]:
df_1.drop(["date_activ",	"date_end",	"date_modif_prod",	"date_renewal"], inplace=True, axis=1)

In [17]:
df_1.head()

,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,-0.006192,0.162916,22034.0,0.0,3084.0,729.06,425.0,0.0,138.95,0.116900,...,0.000011,0.000003,4.860000e-10,0.0,0.0,0.0,0.000011,0.000003,4.860000e-10,0.0
1,-0.004104,0.177779,4060.0,0.0,0.0,597.77,0.0,0.0,6.84,0.142065,...,0.000003,0.000000,0.000000e+00,0.0,0.0,0.0,0.000003,0.000000,0.000000e+00,0.0
2,0.050443,1.500000,7440.0,0.0,1062.0,1311.16,1062.0,30.0,18.37,0.199230,...,0.000003,0.000000,0.000000e+00,0.0,0.0,0.0,0.000003,0.000000,0.000000e+00,0.0
4,-0.003994,-0.000001,11272.0,0.0,0.0,1671.41,0.0,0.0,18.27,0.144149,...,0.000003,0.000000,0.000000e+00,0.0,0.0,0.0,0.000003,0.000000,0.000000e+00,0.0
6,-0.006171,0.000000,267414.0,0.0,19394.0,3077.34,1760.0,0.0,144.86,0.118636,...,0.000011,0.000003,4.860000e-10,0.0,0.0,0.0,0.000011,0.000003,4.860000e-10,0.0


In [18]:
df_1.churn.value_counts()

,count
churn,
0.0,13187
1.0,1419


In [19]:
df.has_gas.value_counts()

,count
has_gas,
f,11955
t,2651


In [20]:
# convert has_gas into encoder

df_1['has_gas'] = df_1['has_gas'].apply(lambda x: 1 if x == 't' else 0)

In [21]:
df_1.select_dtypes(include='object').columns

Index([], dtype='object')

In [22]:
# convert churn into int
df_1['churn'] = df_1['churn'].astype(int)

In [23]:
df_1.head()

,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,-0.006192,0.162916,22034.0,0.0,3084.0,729.06,425.0,0.0,138.95,0.116900,...,0.000011,0.000003,4.860000e-10,0.0,0.0,0.0,0.000011,0.000003,4.860000e-10,0
1,-0.004104,0.177779,4060.0,0.0,0.0,597.77,0.0,0.0,6.84,0.142065,...,0.000003,0.000000,0.000000e+00,0.0,0.0,0.0,0.000003,0.000000,0.000000e+00,0
2,0.050443,1.500000,7440.0,0.0,1062.0,1311.16,1062.0,30.0,18.37,0.199230,...,0.000003,0.000000,0.000000e+00,0.0,0.0,0.0,0.000003,0.000000,0.000000e+00,0
4,-0.003994,-0.000001,11272.0,0.0,0.0,1671.41,0.0,0.0,18.27,0.144149,...,0.000003,0.000000,0.000000e+00,0.0,0.0,0.0,0.000003,0.000000,0.000000e+00,0
6,-0.006171,0.000000,267414.0,0.0,19394.0,3077.34,1760.0,0.0,144.86,0.118636,...,0.000011,0.000003,4.860000e-10,0.0,0.0,0.0,0.000011,0.000003,4.860000e-10,0


In [24]:
X_train, X_test, y_train, y_test = train_test_split(df_1.drop('churn', axis=1), df_1['churn'], test_size=0.2, random_state=42, stratify=df_1['churn'])

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

X_train shape: (11684, 38)
X_test shape: (2922, 38)
y_train shape: (11684,)
y_test shape: (2922,)


In [25]:
X_train.head()

,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,...,var_year_price_mid_peak,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak
7577,-0.009328,0.162916,17750.0,0.0,2915.0,1764.81,2915.0,0.0,146.07,0.115237,...,21.974440,0.000013,2.679260e-06,1.538567e-07,0.007077,0.002548,0.001133,0.007090,2.550800e-03,1.132791e-03
6159,-0.006299,-0.000001,21515.0,0.0,3562.0,550.44,624.0,0.0,105.30,0.142995,...,0.000000,0.000007,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000007,0.000000e+00,0.000000e+00
13298,0.116318,0.162916,139855.0,17967.0,17536.0,6116.50,11474.0,0.0,123.67,0.113323,...,0.001463,0.000011,2.896760e-06,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10
14321,-0.004679,0.177779,16341.0,0.0,0.0,2066.10,0.0,0.0,17.49,0.162033,...,0.000000,0.000004,9.450150e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00
1605,-0.005360,0.236694,23349.0,10284.0,3661.0,2026.80,3661.0,0.0,131.76,0.102772,...,0.000170,0.000006,1.617204e-06,0.000000e+00,0.000000,0.000000,0.000000,0.000006,1.617204e-06,0.000000e+00


In [26]:
df_1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14606 entries, 0 to 16095
Data columns (total 39 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   offpeak_diff_dec_january_energy  14606 non-null  float64
 1   offpeak_diff_dec_january_power   14606 non-null  float64
 2   cons_12m                         14606 non-null  float64
 3   cons_gas_12m                     14606 non-null  float64
 4   cons_last_month                  14606 non-null  float64
 5   forecast_cons_12m                14606 non-null  float64
 6   forecast_cons_year               14606 non-null  float64
 7   forecast_discount_energy         14606 non-null  float64
 8   forecast_meter_rent_12m          14606 non-null  float64
 9   forecast_price_energy_off_peak   14606 non-null  float64
 10  forecast_price_energy_peak       14606 non-null  float64
 11  forecast_price_pow_off_peak      14606 non-null  float64
 12  has_gas                

# Tunning using OPTUNA Library

In [27]:
!pip install optunahub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 856.7/856.7 kB 29.1 MB/s eta 0:00:00
  Attempting uninstall: pyjwt
    Found existing installation: PyJWT 2.3.0
    Uninstalling PyJWT-2.3.0:
      Successfully uninstalled PyJWT-2.3.0


In [28]:
import optunahub

In [29]:
def objective(trial: optuna.Trial):
    classifier_name = trial.suggest_categorical('classifier', ['RandomForest'])
    n_estimators = trial.suggest_int('n_estimators', 50, 1000, step=1)
    max_depth = trial.suggest_int('max_depth', 10, 100, step=1)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20, step=1)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20, step=1)
    min_weight_fraction_leaf = trial.suggest_float('min_weight_fraction_leaf', 0.0, 0.5)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 2, 500, step=1)
    min_impurity_decrease = trial.suggest_float('min_impurity_decrease', 0.0, 0.5)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])
    oob_score = trial.suggest_categorical('oob_score', [True, False])
    if oob_score and not bootstrap:
        oob_score = False

    if classifier_name == 'RandomForest':
        classifier = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            min_weight_fraction_leaf=min_weight_fraction_leaf,
            max_features=max_features,
            max_leaf_nodes=max_leaf_nodes,
            min_impurity_decrease=min_impurity_decrease,
            bootstrap=bootstrap,
            oob_score=oob_score,
            random_state=42
        )

    # Use cross-validation score
    try:
        score = cross_val_score(classifier, X_train, y_train, cv=3, scoring='accuracy')  # Use a few folds
        return score.mean() # Return the mean of the scores
    except ValueError as e:
        print(f"ValueError in trial: {e}")
        return float('-inf')

# Create an Optuna study and optimize the objective function
module = optunahub.load_module(package="samplers/tpe_tutorial")

# Initialize the CustomizableTPESampler
sampler = module.CustomizableTPESampler()

# Create an Optuna study with the sampler
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=133)

# Print the best parameters and the corresponding accuracy
print("Best parameters:", study.best_params)
print("Best accuracy:", study.best_value)



/usr/local/lib/python3.11/dist-packages/optuna/_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2025-03-30 10:08:38,753] A new study created in memory with name: no-name-554fd23e-5b00-4719-8bf7-500cd8b0b952
[I 2025-03-30 10:08:46,891] Trial 0 finished with value: 0.9028586159675561 and parameters: {'classifier': 'RandomForest', 'n_estimators': 926, 'max_depth': 41, 'min_samples_split': 7, 'min_samples_leaf': 5, 'min_weight_fraction_leaf': 0.43965035780731215, 'max_features': 'log2', 'max_leaf_nodes': 3, 'min_impurity_decrease': 0.18553144269102417, 'bootstrap': True, 'oob_score': False}. Best is trial 0 with value: 0.9028586159675561.
[I 2025-03-30 10:08:47,957] Trial 1 finished with value: 0.9028586159675561 and parameters: {'classifier': 'RandomForest', 'n_estimators': 114, 'max_depth': 30, 'min_samples_split': 17, 'min_samples_leaf': 9, 'min_weight_fraction_leaf': 0.35015418286

Best parameters: {'classifier': 'RandomForest', 'n_estimators': 926, 'max_depth': 41, 'min_samples_split': 7, 'min_samples_leaf': 5, 'min_weight_fraction_leaf': 0.43965035780731215, 'max_features': 'log2', 'max_leaf_nodes': 3, 'min_impurity_decrease': 0.18553144269102417, 'bootstrap': True, 'oob_score': False}
Best accuracy: 0.9028586159675561


In [30]:
print("Best parameters:", study.best_params)
print("Best accuracy:", study.best_value)

Best parameters: {'classifier': 'RandomForest', 'n_estimators': 926, 'max_depth': 41, 'min_samples_split': 7, 'min_samples_leaf': 5, 'min_weight_fraction_leaf': 0.43965035780731215, 'max_features': 'log2', 'max_leaf_nodes': 3, 'min_impurity_decrease': 0.18553144269102417, 'bootstrap': True, 'oob_score': False}
Best accuracy: 0.9028586159675561


In [32]:
np.set_printoptions(precision=3)

In [37]:
classifer = RandomForestClassifier(n_estimators=926, max_depth=41,min_samples_split=7,
                                   min_samples_leaf=5, min_weight_fraction_leaf=0.43965035780731215, max_features="log2",
                                   max_leaf_nodes=3, min_impurity_decrease=0.18553144269102417, bootstrap=True, oob_score=False
                                   ).fit(X_train, y_train)

In [38]:
y_pred = classifer.predict(X_test)

In [39]:
accuracy_score(y_test, y_pred)

0.9028062970568104

In [42]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      1.00      0.95      2638
           1       0.00      0.00      0.00       284

    accuracy                           0.90      2922
   macro avg       0.45      0.50      0.47      2922
weighted avg       0.82      0.90      0.86      2922



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [41]:
jaccard_score(y_test, y_pred, pos_label=0)

np.float64(0.9028062970568104)